# Sport Machine — kézilabda-mozdulatfelismerés (UNIRI-HBD)

Mit csinál ez a notebook:
1. Minden kivágott klipnél megkeresi a végrehajtót a teljes jelenetben, és előtanított
   **YOLO11-pose** modellel kinyeri a 17 testpontot kockánként (→ `poses.npz`).
2. Kiértékel három feladatot, **jelenet szerint csoportosított 5-szörös keresztvalidációval**:
   lövés / nem lövés · felugrásos / helyből / egyéb · mind a 7 kategória.
3. Minden lövésre kapus-szempontú profilt ír (lövőkéz, kartartás, felugrás) → `kapus_profilok.csv`.
4. Végső modelleket ment (`model_*.joblib`, `model_*.pt`) a Macen futtatáshoz.

**Beállítás (egyszer):**
- *Add Input* → a feltöltött `uniri-hbd` adathalmaz (a zip; a Kaggle kibontja).
- *Settings → Accelerator*: **GPU T4 x2** (egyet használunk). *Internet*: **On** (pip + súlyok).
- *Run All*. Kb. 30–50 perc; a kimenet a jobb oldali *Output* fülön (`/kaggle/working`).

Ha megszakad: újraindítva a már kész klipeket a gyorsítótárból (`poses_cache/`) veszi.

Adat: UNIRI-HBD, Ivašić-Kos, M., Pobar, M., University of Rijeka, DOI 10.21227/0g0a-fe06, CC BY 4.0.


In [ ]:
!pip install -q ultralytics imageio-ffmpeg
!nvidia-smi -L || true

## A csomag forrása (a repó `ml/hbd` mappájából generálva)

In [ ]:
import os
os.makedirs('hbd', exist_ok=True)

In [ ]:
%%writefile hbd/__init__.py
"""hbd — kézilabda-mozdulatfelismerés valódi videóból (UNIRI-HBD adathalmaz).

Ez a csomag SZÁNDÉKOSAN független a `backend/handball` szabályrétegeitől:
itt a gép tanul felvételekből, nem kézzel írt küszöbökből ítél. A kimenete
(testpont-sorozat, mozdulat-osztály, lövés-profil) később a backend
"gondolkodó" rétegeinek bemenete lesz.

Modulok:
    dataset     — az adathalmaz bejárása (klip → jelenet → kategória)
    video       — képkockák olvasása ffmpeg-gel (OpenCV-dekóder nélkül)
    locate      — a kivágott klip megkeresése a teljes jelenetben
    pose        — testpont-kinyerés előtanított YOLO-pose modellel (CLI)
    features    — testpont-normalizálás, sorozat, klip-szintű jellemzők
    goalkeeper  — kapus-szempontú lövés-profil (kartartás, lövőkar, felugrás)
    models      — kis időbeli konvolúciós háló (torch)
    train       — keresztvalidált kiértékelés + végső modellek (CLI)

Adatforrás: UNIRI-HBD, Ivašić-Kos, M., Pobar, M., University of Rijeka,
DOI 10.21227/0g0a-fe06, CC BY 4.0.
"""

__version__ = "0.1.0"


In [ ]:
%%writefile hbd/dataset.py
"""Az UNIRI-HBD adathalmaz bejárása.

Szerkezet (a zip kibontva):

    handball_s/
        actions/<kategória>/<jelenet>_act<N>.avi   — a végrehajtóra kivágott klip
        scenes/<kategória>/<jelenet>.mp4|.avi       — a teljes jelenet
        player_detections/<kategória>/<jelenet>.csv — játékos-dobozok kockánként

Egy tanító minta = egy kivágott klip (`actions/`). A klip a jelenet
képpontjaiból van kivágva (a sablon-egyezés 1,000), így a jelenetben
pontosan visszakereshető (lásd `locate`). A kiértékelés a JELENET szerint
csoportosít: ugyanannak a jelenetnek két klipje sosem kerülhet egyszerre a
tanító és a teszt részbe.
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path

# A zip kategória-mappái (a "lövés" mappa neve "shot", nem "shooting").
CATEGORIES = ("crossing", "defence", "dribbling", "jump-shot", "passing", "running", "shot")
SHOT_CATEGORIES = ("jump-shot", "shot")

# Feladatok: név → (osztálynevek, kategória → osztály-index)
TASKS = {
    # lövés vagy nem lövés — a kapus első kérdése
    "binary": (("nem_loves", "loves"), {c: int(c in SHOT_CATEGORIES) for c in CATEGORIES}),
    # felugrásos lövés / helyből lövés / egyéb
    "shot3": (
        ("egyeb", "felugrasos", "helybol"),
        {c: {"jump-shot": 1, "shot": 2}.get(c, 0) for c in CATEGORIES},
    ),
    # mind a hét kategória (a szakirodalommal összevethető)
    "all7": (CATEGORIES, {c: i for i, c in enumerate(CATEGORIES)}),
}

VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv")
_ACT_RE = re.compile(r"_act\d+$")


@dataclass(frozen=True)
class Clip:
    """Egy kivágott mozdulat-klip és a hozzá tartozó jelenet."""

    name: str  # a klip fájlneve kiterjesztés nélkül, pl. "shot_KS_100_act1"
    category: str  # pl. "shot"
    action_path: Path
    scene_path: Path
    detections_path: Path | None

    @property
    def scene_id(self) -> str:
        """Csoport-azonosító a kiértékeléshez: kategória/jelenet."""
        return f"{self.category}/{self.scene_path.stem}"


def find_root(path: str | Path) -> Path:
    """Megkeresi az adathalmaz gyökerét (ahol `actions/` és `scenes/` van).

    Elfogadja magát a gyökeret, a `handball_s` szülőjét, vagy egy Kaggle
    bemeneti mappát, ahol a kibontott zip mélyebben ül.
    """
    p = Path(path)
    if (p / "actions").is_dir() and (p / "scenes").is_dir():
        return p
    for cand in sorted(p.rglob("actions")):
        root = cand.parent
        if cand.is_dir() and (root / "scenes").is_dir():
            return root
    raise FileNotFoundError(f"Nem találom az UNIRI-HBD gyökerét (actions/ + scenes/) itt: {p}")


def _find_video(folder: Path, stem: str) -> Path | None:
    for ext in VIDEO_EXTS:
        f = folder / f"{stem}{ext}"
        if f.exists():
            return f
    return None


def index_dataset(root: str | Path, categories: tuple[str, ...] | None = None) -> list[Clip]:
    """Az összes kivágott klip listája, determinisztikus (név szerinti) sorrendben.

    A jelenet nélküli ("árva") klipeket kihagyja — ezekből nem kereshető
    vissza a végrehajtó helye.
    """
    root = find_root(root)
    cats = categories or CATEGORIES
    clips: list[Clip] = []
    for cat in cats:
        act_dir = root / "actions" / cat
        if not act_dir.is_dir():
            continue
        for act in sorted(act_dir.iterdir()):
            if act.suffix.lower() not in VIDEO_EXTS:
                continue
            scene_stem = _ACT_RE.sub("", act.stem)
            scene = _find_video(root / "scenes" / cat, scene_stem)
            if scene is None:
                continue
            det = root / "player_detections" / cat / f"{scene_stem}.csv"
            clips.append(Clip(act.stem, cat, act, scene, det if det.exists() else None))
    return clips


def read_detections(path: str | Path) -> list[list[tuple[float, float, float, float, float]]]:
    """A `player_detections` CSV beolvasása: kockánként a dobozok listája.

    Formátum: "Frame N" sor, alatta soronként "x1, y1, x2, y2, megbízhatóság".
    Követés (ki kicsoda) és a végrehajtó jelölése NINCS benne.
    """
    frames: list[list[tuple[float, float, float, float, float]]] = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("Frame"):
            frames.append([])
            continue
        parts = [float(v) for v in line.split(",")]
        if len(parts) == 4:
            parts.append(1.0)
        if frames and len(parts) == 5:
            frames[-1].append(tuple(parts))  # type: ignore[arg-type]
    return frames


In [ ]:
%%writefile hbd/video.py
"""Képkockák olvasása ffmpeg-gel.

Miért nem OpenCV? A felhős gépen az OpenCV videó-dekódere ezeken a
fájlokon összeomlott (szegmentálási hiba), az ffmpeg viszont stabil és a
Kaggle-képen is megvan. Ha nincs rendszer-ffmpeg, az `imageio-ffmpeg`
csomag statikus binárisát használjuk.

A kockákat EGYENKÉNT adjuk vissza (generátor): egy 2704×1520-as jelenet
egészében több GB lenne a memóriában.
"""

from __future__ import annotations

import re
import shutil
import subprocess
from pathlib import Path
from typing import Iterator

import numpy as np

_SIZE_RE = re.compile(r"\b(\d{2,5})x(\d{2,5})\b")
_FPS_RE = re.compile(r"(\d+(?:\.\d+)?) fps")


def ffmpeg_exe() -> str:
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    try:
        import imageio_ffmpeg  # type: ignore

        return imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError as e:  # pragma: no cover - környezetfüggő
        raise RuntimeError("Nincs ffmpeg: telepítsd (apt) vagy `pip install imageio-ffmpeg`") from e


def probe(path: str | Path) -> tuple[int, int, float]:
    """(szélesség, magasság, fps) az ffmpeg fejléc-kiírásából (ffprobe nélkül)."""
    p = subprocess.run(
        [ffmpeg_exe(), "-hide_banner", "-i", str(path)], capture_output=True, text=True
    )
    for line in p.stderr.splitlines():
        if "Video:" in line:
            m = _SIZE_RE.search(line.split("Video:", 1)[1])
            f = _FPS_RE.search(line)
            if m:
                return int(m.group(1)), int(m.group(2)), float(f.group(1)) if f else 0.0
    raise RuntimeError(f"Nem olvasható videó: {path}")


def iter_frames(path: str | Path, gray: bool = False) -> Iterator[np.ndarray]:
    """A videó kockái sorban: (H, W) szürke vagy (H, W, 3) BGR uint8."""
    w, h, _ = probe(path)
    ch = 1 if gray else 3
    size = w * h * ch
    proc = subprocess.Popen(
        [ffmpeg_exe(), "-v", "error", "-i", str(path), "-f", "rawvideo",
         "-pix_fmt", "gray" if gray else "bgr24", "-"],
        stdout=subprocess.PIPE,
        # a klip végén szándékosan bezárjuk a csövet → az ffmpeg "Broken pipe"-ot
        # írna; ez nem hiba, ne szemetelje a naplót
        stderr=subprocess.DEVNULL,
    )
    assert proc.stdout is not None
    try:
        while True:
            buf = proc.stdout.read(size)
            if len(buf) < size:
                break
            arr = np.frombuffer(buf, np.uint8)
            yield arr.reshape(h, w) if gray else arr.reshape(h, w, 3)
    finally:
        proc.stdout.close()
        proc.kill()
        proc.wait()


def read_frames(path: str | Path, gray: bool = False) -> np.ndarray:
    """Az összes kocka egy tömbben — csak KIS videóra (a kivágott klipekre)."""
    frames = list(iter_frames(path, gray=gray))
    if not frames:
        raise RuntimeError(f"Üres videó: {path}")
    return np.stack(frames)


In [ ]:
%%writefile hbd/locate.py
"""A kivágott klip visszakeresése a teljes jelenetben.

Az UNIRI-HBD `actions/` klipjei a jelenet képpontjainak szó szerinti
kivágásai (állandó méretű ablak, ami a játékost követi). Ezért a klip
k-adik kockája a jelenet (eltolás + k)-adik kockájában sablon-egyezéssel
pontosan megtalálható. Így tudjuk meg, MELYIK játékos a végrehajtó és HOL
áll — a `player_detections` ezt nem mondja meg.

Két lépés:
  1. eltolás: a klip első kockáját a jelenet kockáin keressük, az első
     (gyakorlatilag) pontos egyezésnél megállunk;
  2. követés: a további kockákon csak az előző hely környezetében
     keresünk (gyors), rossz egyezésnél az egész képen.
"""

from __future__ import annotations

from typing import Iterable, Iterator

import cv2
import numpy as np

# A klip a jelenet képpontjainak másolata → a helyes kockán az egyezés 1,000.
# Ennél jobb egyezésnél az eltolás-keresés megállhat. SZIGORÚ kell: álló
# kameránál a SZOMSZÉD kocka is ~0,98-at ad, és 0,98-as küszöbnél a keresés
# egy kockával korán megállt (az egész klip elcsúszott, 0,85-ös egyezésekkel).
EXACT_MATCH = 0.999
# Követéskor ennél gyengébb ablakos egyezésnél az egész képen keresünk.
TRACK_MIN_SCORE = 0.90
# A keresőablak a sablon körül: a sablon-méret ennyiszerese, de min. ennyi px.
SEARCH_PAD_FRAC = 0.6
SEARCH_PAD_MIN = 32


def _gray(frame: np.ndarray) -> np.ndarray:
    return frame if frame.ndim == 2 else cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)


def match_full(frame: np.ndarray, tmpl: np.ndarray) -> tuple[float, tuple[int, int]]:
    """Legjobb egyezés az egész képen: (pontszám, (x, y) bal-felső sarok)."""
    r = cv2.matchTemplate(_gray(frame), tmpl, cv2.TM_CCOEFF_NORMED)
    _, score, _, loc = cv2.minMaxLoc(r)
    return float(score), (int(loc[0]), int(loc[1]))


def match_window(
    frame: np.ndarray, tmpl: np.ndarray, prev_xy: tuple[int, int]
) -> tuple[float, tuple[int, int]]:
    """Egyezés az előző hely környezetében; ha az ablak nem fér, az egész képen."""
    g = _gray(frame)
    th, tw = tmpl.shape
    pad = max(SEARCH_PAD_MIN, int(SEARCH_PAD_FRAC * max(th, tw)))
    x0 = max(0, prev_xy[0] - pad)
    y0 = max(0, prev_xy[1] - pad)
    x1 = min(g.shape[1], prev_xy[0] + tw + pad)
    y1 = min(g.shape[0], prev_xy[1] + th + pad)
    if x1 - x0 < tw or y1 - y0 < th:
        return match_full(g, tmpl)
    r = cv2.matchTemplate(g[y0:y1, x0:x1], tmpl, cv2.TM_CCOEFF_NORMED)
    _, score, _, loc = cv2.minMaxLoc(r)
    return float(score), (int(loc[0]) + x0, int(loc[1]) + y0)


def find_offset(frames: Iterable[np.ndarray], first_tmpl: np.ndarray) -> tuple[int, float]:
    """A jelenet melyik kockájánál kezdődik a klip: (eltolás, pontszám).

    Az első pontos egyezésnél megáll; ha nincs pontos, a legjobbat adja.
    """
    best = (-1, -1.0)
    for i, fr in enumerate(frames):
        score, _ = match_full(fr, first_tmpl)
        if score > best[1]:
            best = (i, score)
        if score >= EXACT_MATCH:
            break
    return best


def track(
    frames: Iterable[np.ndarray], action_gray: np.ndarray, offset: int
) -> Iterator[tuple[int, np.ndarray, tuple[int, int, int, int], float]]:
    """Végigköveti a klipet a jeleneten.

    Bemenet: a jelenet kockái (szürke vagy BGR, sorban), a klip szürke kockái
    (T, h, w) és az eltolás. Kimenet kockánként: (k, a jelenet kockája,
    doboz (x1, y1, x2, y2), pontszám). A jelenet végén megáll.
    """
    th, tw = action_gray.shape[1:]
    prev: tuple[int, int] | None = None
    for i, fr in enumerate(frames):
        k = i - offset
        if k < 0:
            continue
        if k >= len(action_gray):
            break
        tmpl = action_gray[k]
        if prev is None:
            score, xy = match_full(fr, tmpl)
        else:
            score, xy = match_window(fr, tmpl, prev)
            if score < TRACK_MIN_SCORE:
                s2, xy2 = match_full(fr, tmpl)
                if s2 > score:
                    score, xy = s2, xy2
        prev = xy
        yield k, fr, (xy[0], xy[1], xy[0] + tw, xy[1] + th), score


In [ ]:
%%writefile hbd/pose.py
"""Testpont-kinyerés: minden klip végrehajtójára 17 COCO-testpont kockánként.

Folyamat klipenként:
  1. a kivágott klip megkeresése a jelenetben (`locate`) → a végrehajtó doboza,
  2. a doboz körül négyzetes, MARGÓS kivágás a teljes felbontású jelenetből
     (a pózmodellnek kell a környezet; a klip maga szorosan vágott),
  3. előtanított YOLO-pose (Ultralytics) a kivágásokon, kötegben,
  4. a kivágáson talált emberek közül az, amelyik a végrehajtó dobozával a
     legjobban fed (több ember is lehet a kivágásban),
  5. a testpontok visszaszámolása a jelenet képkoordinátáiba.

Kimenet: egyetlen tömör .npz (néhány MB) — erre épül minden tanítás.
Megszakítás után folytatható: klipenként gyorsítótár-fájl készül.

Futtatás (Kaggle GPU-n):
    python -m hbd.pose --data /kaggle/input/uniri-hbd --out /kaggle/working/poses.npz --device 0
Gyors próba CPU-n:
    python -m hbd.pose --data ./handball_s --out poses.npz --limit 5 --weights yolo11n-pose.pt
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

import numpy as np

from .dataset import Clip, index_dataset
from .locate import find_offset, track
from .video import iter_frames, probe, read_frames

KP_COUNT = 17
# A pózkivágás oldala = a végrehajtó dobozának hosszabbik oldala × ennyi.
CROP_SCALE = 1.6
# A pózmodell bemeneti mérete (a ~110 px-es játékost erre nagyítjuk).
POSE_IMGSZ = 384
# Ennél kisebb fedés (IoU) a végrehajtó dobozával → nem őt találta a modell.
MIN_PICK_IOU = 0.15
# Alapértelmezett súlyok: GPU-n a legpontosabb, CPU-n a legkisebb.
DEFAULT_WEIGHTS_GPU = "yolo11x-pose.pt"
DEFAULT_WEIGHTS_CPU = "yolo11n-pose.pt"

# COCO-csontváz a rajzoláshoz
SKELETON = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12),
            (11, 13), (13, 15), (12, 14), (14, 16), (0, 5), (0, 6)]


def square_crop_box(box, frame_w: int, frame_h: int, scale: float = CROP_SCALE):
    """Négyzetes kivágó-ablak a doboz közepe körül, a képre szorítva: (x0, y0, x1, y1)."""
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * scale
    x0 = int(round(max(0, cx - side / 2)))
    y0 = int(round(max(0, cy - side / 2)))
    xe = int(round(min(frame_w, cx + side / 2)))
    ye = int(round(min(frame_h, cy + side / 2)))
    return x0, y0, xe, ye


def iou(a, b) -> float:
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def pick_person(det_boxes: np.ndarray, target_box) -> int | None:
    """A végrehajtó dobozával legjobban fedő észlelés indexe (vagy None)."""
    if len(det_boxes) == 0:
        return None
    scores = [iou(b, target_box) for b in det_boxes]
    j = int(np.argmax(scores))
    return j if scores[j] >= MIN_PICK_IOU else None


class PoseModel:
    """Vékony burok az Ultralytics YOLO-pose köré (lusta import)."""

    def __init__(self, weights: str, device: str | None = None, imgsz: int = POSE_IMGSZ,
                 batch: int = 32):
        from ultralytics import YOLO  # lusta: a tesztekhez nem kell

        self.model = YOLO(weights)
        self.device = device
        self.imgsz = imgsz
        self.batch = batch

    def __call__(self, crops: list[np.ndarray]) -> list[tuple[np.ndarray, np.ndarray]]:
        """Kivágásonként: (dobozok (n, 4), testpontok (n, 17, 3)) a kivágás koordinátáiban."""
        out = []
        for i in range(0, len(crops), self.batch):
            res = self.model.predict(crops[i:i + self.batch], imgsz=self.imgsz,
                                     device=self.device, verbose=False, conf=0.1)
            for r in res:
                if r.keypoints is None or len(r.boxes) == 0:
                    out.append((np.zeros((0, 4), np.float32), np.zeros((0, KP_COUNT, 3), np.float32)))
                    continue
                boxes = r.boxes.xyxy.cpu().numpy().astype(np.float32)
                xy = r.keypoints.xy.cpu().numpy()
                conf = r.keypoints.conf
                conf = conf.cpu().numpy() if conf is not None else np.ones(xy.shape[:2])
                out.append((boxes, np.concatenate([xy, conf[..., None]], -1).astype(np.float32)))
        return out


def extract_clip(clip: Clip, model) -> dict:
    """Egy klip végrehajtójának testpont-sorozata a jelenet koordinátáiban."""
    action = read_frames(clip.action_path, gray=True)
    w, h, fps = probe(clip.scene_path)
    offset, off_score = find_offset(iter_frames(clip.scene_path, gray=True), action[0])

    crops, origins, boxes, scores = [], [], [], []
    for _k, frame, box, score in track(iter_frames(clip.scene_path), action, offset):
        x0, y0, x1, y1 = square_crop_box(box, w, h)
        crops.append(np.ascontiguousarray(frame[y0:y1, x0:x1]))
        origins.append((x0, y0))
        boxes.append(box)
        scores.append(score)

    T = len(crops)
    kps = np.zeros((T, KP_COUNT, 3), np.float32)
    if T:
        for t, (det_boxes, det_kps) in enumerate(model(crops)):
            ox, oy = origins[t]
            tb = boxes[t]
            local_target = (tb[0] - ox, tb[1] - oy, tb[2] - ox, tb[3] - oy)
            j = pick_person(det_boxes, local_target)
            if j is not None:
                kps[t] = det_kps[j]
                kps[t, :, 0] += ox
                kps[t, :, 1] += oy
    return {
        "kps": kps,
        "boxes": np.asarray(boxes, np.float32).reshape(T, 4),
        "match": np.asarray(scores, np.float32),
        "offset": offset,
        "offset_score": off_score,
        "fps": fps,
        "frame_size": (w, h),
        "action_len": len(action),
        "crops_preview": crops[T // 2] if T else None,
    }


def save_dataset(path: Path, clips: list[Clip], results: list[dict]) -> None:
    """Az összes klip egy tömör .npz-ben (pickle nélkül)."""
    lens = [len(r["kps"]) for r in results]
    offsets = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)
    np.savez_compressed(
        path,
        names=np.array([c.name for c in clips]),
        categories=np.array([c.category for c in clips]),
        scene_ids=np.array([c.scene_id for c in clips]),
        offsets=offsets,
        kps=np.concatenate([r["kps"] for r in results]) if results else np.zeros((0, KP_COUNT, 3)),
        boxes=np.concatenate([r["boxes"] for r in results]) if results else np.zeros((0, 4)),
        match=np.concatenate([r["match"] for r in results]) if results else np.zeros(0),
        fps=np.array([r["fps"] for r in results], np.float32),
        frame_size=np.array([r["frame_size"] for r in results], np.int32).reshape(-1, 2),
    )


def load_dataset(path: str | Path) -> dict:
    """Az .npz visszaolvasása klip-listává: [{name, category, scene_id, kps, boxes, fps}]."""
    z = np.load(path)
    off = z["offsets"]
    clips = []
    for i, name in enumerate(z["names"]):
        a, b = off[i], off[i + 1]
        clips.append({
            "name": str(name),
            "category": str(z["categories"][i]),
            "scene_id": str(z["scene_ids"][i]),
            "kps": z["kps"][a:b],
            "boxes": z["boxes"][a:b],
            "match": z["match"][a:b],
            "fps": float(z["fps"][i]) or 30.0,
        })
    return {"clips": clips}


def draw_pose(img: np.ndarray, kps: np.ndarray, min_conf: float = 0.3) -> np.ndarray:
    import cv2

    out = img.copy()
    for a, b in SKELETON:
        if kps[a, 2] >= min_conf and kps[b, 2] >= min_conf:
            cv2.line(out, tuple(int(v) for v in kps[a, :2]), tuple(int(v) for v in kps[b, :2]),
                     (0, 255, 255), 1, cv2.LINE_AA)
    for j, (x, y, c) in enumerate(kps):
        if c >= min_conf:
            col = (0, 0, 255) if j in (9, 10) else (0, 255, 0)  # csukló piros
            cv2.circle(out, (int(x), int(y)), 2, col, -1, cv2.LINE_AA)
    return out


def _preview_tile(clip: Clip, res: dict, size: int = 200) -> np.ndarray | None:
    import cv2

    T = len(res["kps"])
    if T == 0:
        return None
    t = T // 2
    crop = res["crops_preview"]
    w, h = res["frame_size"]
    x0, y0, _, _ = square_crop_box(res["boxes"][t], w, h)
    k = res["kps"][t].copy()
    k[:, 0] -= x0
    k[:, 1] -= y0
    sc = size / max(crop.shape[:2])
    img = cv2.resize(crop, None, fx=sc, fy=sc, interpolation=cv2.INTER_CUBIC)
    k[:, :2] *= sc
    img = draw_pose(img, k)
    tile = np.zeros((size + 18, size, 3), np.uint8)
    tile[:img.shape[0], :img.shape[1]] = img
    cv2.putText(tile, clip.name[:28], (2, size + 13), cv2.FONT_HERSHEY_SIMPLEX, 0.33,
                (255, 255, 255), 1, cv2.LINE_AA)
    return tile


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description="UNIRI-HBD: testpont-kinyerés a végrehajtóra")
    ap.add_argument("--data", required=True, help="az adathalmaz (vagy egy szülő mappája)")
    ap.add_argument("--out", required=True, help="kimeneti .npz")
    ap.add_argument("--weights", default=None, help="YOLO-pose súlyok (alap: GPU-n x, CPU-n n)")
    ap.add_argument("--device", default=None, help="pl. 0 (GPU) vagy cpu")
    ap.add_argument("--imgsz", type=int, default=POSE_IMGSZ)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--limit", type=int, default=0, help="csak az első N klip kategóriánként")
    ap.add_argument("--categories", default="", help="vesszővel, pl. shot,jump-shot")
    ap.add_argument("--preview", type=int, default=0, help="ennyi ellenőrző kép kategóriánként")
    args = ap.parse_args(argv)

    cats = tuple(c for c in args.categories.split(",") if c) or None
    clips = index_dataset(args.data, cats)
    if args.limit:
        per: dict[str, int] = {}
        kept = []
        for c in clips:
            per[c.category] = per.get(c.category, 0) + 1
            if per[c.category] <= args.limit:
                kept.append(c)
        clips = kept

    weights = args.weights
    if weights is None:
        try:
            import torch

            gpu = torch.cuda.is_available() and args.device != "cpu"
        except ImportError:
            gpu = False
        weights = DEFAULT_WEIGHTS_GPU if gpu else DEFAULT_WEIGHTS_CPU
    model = PoseModel(weights, device=args.device, imgsz=args.imgsz, batch=args.batch)

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    cache = out.parent / (out.stem + "_cache")
    cache.mkdir(exist_ok=True)

    results, done_clips, tiles = [], [], {}
    t0 = time.time()
    for i, clip in enumerate(clips):
        cf = cache / f"{clip.category}__{clip.name}.npz"
        want_tile = len(tiles.get(clip.category, [])) < args.preview
        if cf.exists() and not want_tile:
            z = np.load(cf)
            res = {k: z[k] for k in ("kps", "boxes", "match")}
            res.update(fps=float(z["fps"]), frame_size=tuple(z["frame_size"]))
        else:
            try:
                res = extract_clip(clip, model)
            except Exception as e:  # egy hibás videó ne vigye el a futást
                print(f"  ! {clip.name}: {e}", file=sys.stderr)
                continue
            np.savez_compressed(cf, kps=res["kps"], boxes=res["boxes"], match=res["match"],
                                fps=res["fps"], frame_size=np.array(res["frame_size"]))
            if want_tile:
                tile = _preview_tile(clip, res)
                if tile is not None:
                    tiles.setdefault(clip.category, []).append(tile)
        results.append(res)
        done_clips.append(clip)
        if (i + 1) % 25 == 0 or i + 1 == len(clips):
            el = time.time() - t0
            print(f"  {i + 1}/{len(clips)} klip, {el:.0f} s (≈{el / (i + 1):.1f} s/klip)", flush=True)

    save_dataset(out, done_clips, results)
    total = sum(len(r["kps"]) for r in results)
    seen = sum(int((r["kps"][:, :, 2].max(1) > 0).sum()) for r in results)
    low = sum(int((r["match"] < 0.9).sum()) for r in results)
    print(f"Kész: {len(done_clips)} klip, {total} kocka → {out}")
    print(f"  pózt talált a kockák {100 * seen / max(1, total):.1f}%-án; "
          f"gyenge klip-egyezés (<0,9): {low} kocka")

    if tiles:
        import cv2

        rows = []
        width = max(len(v) for v in tiles.values())
        for cat in sorted(tiles):
            row = tiles[cat] + [np.zeros_like(tiles[cat][0])] * (width - len(tiles[cat]))
            rows.append(np.hstack(row))
        pv = out.parent / (out.stem + "_preview.jpg")
        cv2.imwrite(str(pv), np.vstack(rows))
        print(f"  ellenőrző kép: {pv}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/features.py
"""Testpont-sorozatból tanítható bemenet.

Két alak:
  * SOROZAT (a neurális hálónak): kockánként a 17 testpont a csípő-
    középponthoz viszonyítva, törzshosszal osztva, felfelé pozitív y-nal,
    plusz a test középpontjának elmozdulása; fix hosszra (SEQ_LEN) átmintázva.
  * JELLEMZŐK (a döntési fának): klipenként néhány tucat értelmezhető szám
    (kar-magasság, csukló-sebesség, felugrás, váll-szélesség, ...).

A normalizálás teszi a kis (~110 px) és a nagy játékost, a kép bal és jobb
szélét összevethetővé. Az idő MÁSODPERCBEN számít (a jelenetek 30 és 48
fps-esek vegyesen) — a sebesség tehát egység / s, nem egység / kocka.
"""

from __future__ import annotations

import numpy as np

# COCO testpont-indexek
NOSE = 0
L_SH, R_SH, L_EL, R_EL, L_WR, R_WR = 5, 6, 7, 8, 9, 10
L_HIP, R_HIP, L_KN, R_KN, L_AN, R_AN = 11, 12, 13, 14, 15, 16
# Tükrözéskor (bal ↔ jobb) a testpontok cseréje
FLIP_PERM = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]

# Ennél kisebb megbízhatóságú testpontot hiányzónak veszünk és időben pótoljuk.
KP_MIN_CONF = 0.3
# A neurális háló bemenetének hossza (kocka).
SEQ_LEN = 32
# A sorozat csatornái: 17×(x, y) + középpont-elmozdulás (dx, dy) + 17 megbízhatóság
SEQ_CHANNELS = 17 * 2 + 2 + 17


def fill_missing(kps: np.ndarray, min_conf: float = KP_MIN_CONF):
    """Hiányzó testpontok időbeli lineáris pótlása.

    Visszaad: xy (T, 17, 2) — a sosem látott testpont NaN —, valid (T, 17).
    """
    T = len(kps)
    xy = kps[..., :2].astype(np.float64).copy()
    valid = kps[..., 2] >= min_conf
    t = np.arange(T)
    for j in range(kps.shape[1]):
        v = valid[:, j]
        if v.sum() == 0:
            xy[:, j] = np.nan
        elif v.sum() < T:
            for d in range(2):
                xy[:, j, d] = np.interp(t, t[v], xy[v, j, d])
    return xy, valid


def _mid(xy, a, b):
    return (xy[:, a] + xy[:, b]) / 2


def body_scale(xy: np.ndarray, boxes: np.ndarray | None = None) -> float:
    """Törzshossz px-ben (váll-közép ↔ csípő-közép medián); ha nincs, a doboz-magasság 1/3-a."""
    torso = np.linalg.norm(_mid(xy, L_SH, R_SH) - _mid(xy, L_HIP, R_HIP), axis=1)
    s = float(np.nanmedian(torso)) if np.isfinite(torso).any() else float("nan")
    if not np.isfinite(s) or s < 1e-3:
        if boxes is not None and len(boxes):
            s = float(np.median(boxes[:, 3] - boxes[:, 1])) / 3
        else:
            s = 1.0
    return max(s, 1e-3)


def normalize(kps: np.ndarray, boxes: np.ndarray | None = None):
    """Testközpontú, törzshosszal osztott, felfelé pozitív koordináták.

    Visszaad: rel (T, 17, 2), center_motion (T, 2) a kezdőponthoz képest,
    conf (T, 17), scale (px).
    """
    xy, _valid = fill_missing(kps)
    scale = body_scale(xy, boxes)
    center = _mid(xy, L_HIP, R_HIP)
    if boxes is not None and len(boxes):
        box_c = np.stack([(boxes[:, 0] + boxes[:, 2]) / 2, (boxes[:, 1] + boxes[:, 3]) / 2], 1)
        bad = ~np.isfinite(center).all(1)
        center[bad] = box_c[bad]
    center = np.nan_to_num(center)
    rel = (xy - center[:, None, :]) / scale
    rel[..., 1] *= -1  # kép-y lefelé nő → felfelé pozitívra fordítjuk
    motion = (center - center[:1]) / scale
    motion[:, 1] *= -1
    return np.nan_to_num(rel), motion, kps[..., 2].astype(np.float64), scale


def resample(arr: np.ndarray, n: int = SEQ_LEN) -> np.ndarray:
    """Időbeli lineáris átmintázás n kockára (az első tengely az idő)."""
    T = len(arr)
    if T == 0:
        return np.zeros((n,) + arr.shape[1:])
    if T == 1:
        return np.repeat(arr, n, axis=0)
    src = np.linspace(0, T - 1, n)
    i0 = np.floor(src).astype(int)
    i1 = np.minimum(i0 + 1, T - 1)
    w = (src - i0).reshape((-1,) + (1,) * (arr.ndim - 1))
    return arr[i0] * (1 - w) + arr[i1] * w


def sequence(kps: np.ndarray, boxes: np.ndarray | None = None, n: int = SEQ_LEN) -> np.ndarray:
    """A neurális háló bemenete: (n, SEQ_CHANNELS) float32."""
    rel, motion, conf, _ = normalize(kps, boxes)
    T = len(kps)
    flat = np.concatenate([rel.reshape(T, -1), motion, conf], axis=1)
    return resample(flat, n).astype(np.float32)


def flip_sequence(seq: np.ndarray) -> np.ndarray:
    """Tükrözött sorozat (adatbővítéshez): x negálva, bal ↔ jobb testpontok cserélve."""
    out = seq.copy()
    pts = out[:, :34].reshape(len(out), 17, 2)[:, FLIP_PERM]
    pts[..., 0] *= -1
    out[:, :34] = pts.reshape(len(out), -1)
    out[:, 34] *= -1  # vízszintes elmozdulás
    out[:, 36:] = seq[:, 36:][:, FLIP_PERM]
    return out


def _angle(a, b, c):
    """A b csúcsú szög fokban (a-b-c), kockánként."""
    v1, v2 = a - b, c - b
    cos = (v1 * v2).sum(-1) / (np.linalg.norm(v1, axis=-1) * np.linalg.norm(v2, axis=-1) + 1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def _speed(p: np.ndarray, fps: float) -> np.ndarray:
    """Pont-pálya sebessége egység / s (kockánként, az első kocka 0)."""
    if len(p) < 2:
        return np.zeros(len(p))
    v = np.linalg.norm(np.diff(p, axis=0), axis=-1) * fps
    return np.concatenate([[0.0], v])


def _stats(prefix: str, x: np.ndarray) -> dict:
    if len(x) == 0:
        return {f"{prefix}_{k}": 0.0 for k in ("mean", "std", "min", "max")}
    return {f"{prefix}_mean": float(np.mean(x)), f"{prefix}_std": float(np.std(x)),
            f"{prefix}_min": float(np.min(x)), f"{prefix}_max": float(np.max(x))}


def clip_features(kps: np.ndarray, boxes: np.ndarray | None, fps: float) -> dict:
    """Klip-szintű, értelmezhető jellemzők (név → szám, rögzített sorrend)."""
    fps = fps or 30.0
    rel, motion, conf, _ = normalize(kps, boxes)
    T = len(kps)
    f: dict[str, float] = {
        "duration_s": T / fps,
        "kp_conf_mean": float(conf.mean()) if T else 0.0,
        "kp_valid_frac": float((conf >= KP_MIN_CONF).mean()) if T else 0.0,
    }
    for side, sh, el, wr in (("l", L_SH, L_EL, L_WR), ("r", R_SH, R_EL, R_WR)):
        h = rel[:, wr, 1] - rel[:, sh, 1]  # csukló a váll fölött (+)
        f.update(_stats(f"wrist_h_{side}", h))
        f[f"wrist_h_{side}_argmax"] = float(np.argmax(h) / max(1, T - 1)) if T else 0.0
        f.update(_stats(f"wrist_v_{side}", _speed(rel[:, wr], fps)))
        f.update(_stats(f"elbow_ang_{side}", _angle(rel[:, sh], rel[:, el], rel[:, wr])))
    both_h = np.maximum(rel[:, L_WR, 1] - rel[:, L_SH, 1], rel[:, R_WR, 1] - rel[:, R_SH, 1])
    f["wrist_h_max_both"] = float(both_h.max()) if T else 0.0
    head_y = rel[:, NOSE, 1]
    f["wrist_above_head_frac"] = float(
        (np.maximum(rel[:, L_WR, 1], rel[:, R_WR, 1]) > head_y).mean()) if T else 0.0
    # test-mozgás: vízszintes sebesség, függőleges kilengés (felugrás)
    f.update(_stats("center_v", _speed(motion, fps)))
    f["center_dy_range"] = float(np.ptp(motion[:, 1])) if T else 0.0
    f["center_dx_range"] = float(np.ptp(motion[:, 0])) if T else 0.0
    # váll-szélesség a törzshez: kicsi = oldalt áll a kamerának, nagy = szemben
    f.update(_stats("shoulder_w", np.abs(rel[:, L_SH, 0] - rel[:, R_SH, 0])))
    # törzs dőlése a függőlegestől (fok)
    sh_mid = (rel[:, L_SH] + rel[:, R_SH]) / 2
    lean = np.degrees(np.arctan2(np.abs(sh_mid[:, 0]), np.abs(sh_mid[:, 1]) + 1e-9))
    f.update(_stats("lean", lean))
    f.update(_stats("knee_ang_l", _angle(rel[:, L_HIP], rel[:, L_KN], rel[:, L_AN])))
    f.update(_stats("knee_ang_r", _angle(rel[:, R_HIP], rel[:, R_KN], rel[:, R_AN])))
    f["ankle_spread_max"] = float(np.abs(rel[:, L_AN, 0] - rel[:, R_AN, 0]).max()) if T else 0.0
    return f


def feature_matrix(clips: list[dict]) -> tuple[np.ndarray, list[str]]:
    rows = [clip_features(c["kps"], c.get("boxes"), c.get("fps", 30.0)) for c in clips]
    names = list(rows[0]) if rows else []
    X = np.array([[r[n] for n in names] for r in rows], np.float64)
    return np.nan_to_num(X), names


In [ ]:
%%writefile hbd/goalkeeper.py
"""Kapus-szempontú lövés-profil a testpont-sorozatból.

A kapus azt nézi: melyik kézzel lő, hol a lövőkar az elengedéskor (fent /
oldalt / lent), felugrik-e, merre fordul a teste. Ebből ítéli meg, hova
tud a lövő reálisan lőni. Ez a modul ezeket méri — CÍMKE NÉLKÜL, a
testpontokból. A hova-ment-a-lövés tanításához (következő lépés) ezek
lesznek a bemenet.

Ellenőrizhetőség: a felugrás-ítélet összevethető az adathalmaz címkéjével
(jump-shot = felugrásos, shot = helyből) — ezt a `train` meg is méri. A
kartartásra nincs címke; azt egy kis kézi mintán kell ellenőrizni.

Korlát: a "kamerához képest" szavak szó szerint értendők. A pálya-
koordinátás irányhoz (kapu felé fordulás) kamera-kalibráció kell.
"""

from __future__ import annotations

import numpy as np

from .features import L_HIP, L_SH, L_WR, NOSE, R_HIP, R_SH, R_WR, normalize

# Kartartás az elengedéskor: a csukló magassága a vállhoz képest, törzshosszban.
# Ennél magasabban → "fent" (a fej vonalában vagy fölötte).
ARM_HIGH_MIN = 0.35
# Ennél mélyebben → "lent" (csípő felé); a kettő között → "oldalt".
ARM_LOW_MAX = -0.25
# Felugrás: a csípő ennyi törzshossznyit emelkedik az elengedés körüli
# alapvonal fölé (a futás okozta mélységi elmozdulást a lineáris alapvonal
# kiszűri).
JUMP_MIN_RISE = 0.20
# Az elengedés körüli ablak (másodperc) a felugrás-méréshez — IDŐTARTAM,
# ezért másodpercben, nem kockában (a jelenetek 30 és 48 fps-esek).
JUMP_WINDOW_S = 0.6
# Az alapvonal két végének hossza (másodperc).
JUMP_EDGE_S = 0.12
# Kevés mintánál nincs ítélet (None), nem hallgatólagos alapérték.
MIN_FRAMES = 8
MIN_VALID_FRAC = 0.5


def _smooth(x: np.ndarray) -> np.ndarray:
    if len(x) < 3:
        return x
    return np.convolve(np.pad(x, 1, mode="edge"), np.ones(3) / 3, mode="valid")


def jump_rise(center_up: np.ndarray, release: int, fps: float) -> float:
    """A csípő legnagyobb emelkedése a lineáris alapvonal fölé az elengedés körül."""
    half = max(2, int(round(JUMP_WINDOW_S * fps)))
    edge = max(1, int(round(JUMP_EDGE_S * fps)))
    a = max(0, release - half)
    b = min(len(center_up), release + half + 1)
    seg = center_up[a:b]
    if len(seg) < 2 * edge + 1:
        return 0.0
    y0, y1 = seg[:edge].mean(), seg[-edge:].mean()
    base = np.linspace(y0, y1, len(seg))
    return float(np.max(seg - base))


def shot_profile(kps: np.ndarray, boxes: np.ndarray | None, fps: float) -> dict | None:
    """Egy lövés kapus-szempontú profilja; kevés/rossz mintánál None.

    Kulcsok:
        elengedes_kocka, elengedes_ido_s — a lövőkéz csuklójának leggyorsabb pillanata
        lovokez        — "jobb" / "bal" (a lövő SAJÁT keze, anatómiailag)
        kartartas      — "fent" / "oldalt" / "lent"
        kar_magassag   — a csukló a váll fölött, törzshosszban (+ = fölötte)
        felugras       — bool; csipo_emelkedes — törzshosszban
        vallszelesseg  — |bal váll − jobb váll| / törzs: kicsi = oldalt áll a kamerának
        torzs_doles_fok — a törzs dőlése a függőlegestől
        lovo_helye_px  — (x, y) a két boka közepe a jelenet képén (pálya-kalibrációhoz)
    """
    fps = fps or 30.0
    T = len(kps)
    if T < MIN_FRAMES:
        return None
    valid = kps[..., 2] >= 0.3
    arm_valid = valid[:, [L_SH, R_SH, L_WR, R_WR]].mean()
    if arm_valid < MIN_VALID_FRAC:
        return None
    rel, motion, _conf, _scale = normalize(kps, boxes)

    # Lövőkéz: a csukló testhez mért sebességének csúcsa a nagyobb.
    def wrist_speed(j):
        d = np.linalg.norm(np.diff(rel[:, j], axis=0), axis=1) * fps
        return _smooth(np.concatenate([[0.0], d]))

    vl, vr = wrist_speed(L_WR), wrist_speed(R_WR)
    right = vr.max() >= vl.max()
    wr, sh = (R_WR, R_SH) if right else (L_WR, L_SH)
    release = int(np.argmax(vr if right else vl))

    h = float(rel[release, wr, 1] - rel[release, sh, 1])
    if h >= ARM_HIGH_MIN:
        arm = "fent"
    elif h <= ARM_LOW_MAX:
        arm = "lent"
    else:
        arm = "oldalt"

    rise = jump_rise(motion[:, 1], release, fps)
    sh_w = float(abs(rel[release, L_SH, 0] - rel[release, R_SH, 0]))
    sh_mid = (rel[release, L_SH] + rel[release, R_SH]) / 2
    lean = float(np.degrees(np.arctan2(abs(sh_mid[0]), abs(sh_mid[1]) + 1e-9)))
    feet = kps[release, [15, 16], :2].mean(0)
    if not (kps[release, [15, 16], 2] >= 0.3).all() and boxes is not None and len(boxes) > release:
        b = boxes[release]
        feet = np.array([(b[0] + b[2]) / 2, b[3]])
    return {
        "elengedes_kocka": release,
        "elengedes_ido_s": round(release / fps, 3),
        "lovokez": "jobb" if right else "bal",
        "kartartas": arm,
        "kar_magassag": round(h, 3),
        "fej_folott": bool(rel[release, wr, 1] > rel[release, NOSE, 1]),
        "felugras": bool(rise >= JUMP_MIN_RISE),
        "csipo_emelkedes": round(rise, 3),
        "vallszelesseg": round(sh_w, 3),
        "torzs_doles_fok": round(lean, 1),
        "lovo_helye_px": (round(float(feet[0]), 1), round(float(feet[1]), 1)),
    }


__all__ = ["shot_profile", "jump_rise", "ARM_HIGH_MIN", "ARM_LOW_MAX", "JUMP_MIN_RISE",
           "L_HIP", "R_HIP"]


In [ ]:
%%writefile hbd/models.py
"""Kis időbeli konvolúciós háló a testpont-sorozatra (torch).

Miért ilyen kicsi? ~800 klipünk van: egy nagy háló bemagolná a tanító
részt. Két konvolúciós réteg + átlag/max-összevonás + adatbővítés
(tükrözés, kis nagyítás, időbeli eltolás) a reális méret.
"""

from __future__ import annotations

import numpy as np

from .features import SEQ_CHANNELS, flip_sequence


def _torch():
    import torch  # lusta import: a csomag többi része torch nélkül is fut

    return torch


def build_model(n_classes: int, channels: int = SEQ_CHANNELS, width: int = 64):
    torch = _torch()
    nn = torch.nn

    class TemporalCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(channels, width, 5, padding=2), nn.BatchNorm1d(width), nn.ReLU(),
                nn.Conv1d(width, width * 2, 5, padding=2), nn.BatchNorm1d(width * 2), nn.ReLU(),
                nn.Conv1d(width * 2, width * 2, 3, padding=1), nn.BatchNorm1d(width * 2), nn.ReLU(),
            )
            self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(width * 4, n_classes))

        def forward(self, x):  # x: (B, T, C)
            h = self.net(x.transpose(1, 2))
            return self.head(torch.cat([h.mean(-1), h.amax(-1)], 1))

    return TemporalCNN()


def _augment(batch: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    out = batch.copy()
    for i in range(len(out)):
        if rng.random() < 0.5:
            out[i] = flip_sequence(out[i])
        out[i, :, :36] *= rng.uniform(0.9, 1.1)
        shift = int(rng.integers(-2, 3))
        if shift:
            out[i] = np.roll(out[i], shift, axis=0)
    return out


class SeqClassifier:
    """sklearn-szerű burok: fit(X, y) / predict(X) / predict_proba(X), X: (N, T, C)."""

    def __init__(self, n_classes: int, epochs: int = 80, lr: float = 2e-3, batch: int = 32,
                 seed: int = 0, device: str | None = None):
        self.n_classes, self.epochs, self.lr, self.batch, self.seed = n_classes, epochs, lr, batch, seed
        torch = _torch()
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.mu = self.sd = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        torch = _torch()
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        self.mu = X.reshape(-1, X.shape[-1]).mean(0)
        self.sd = X.reshape(-1, X.shape[-1]).std(0) + 1e-6
        self.model = build_model(self.n_classes, X.shape[-1]).to(self.device)
        counts = np.bincount(y, minlength=self.n_classes).astype(np.float64)
        w = counts.sum() / (self.n_classes * np.maximum(counts, 1))
        loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=self.device))
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-3)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, self.epochs)
        self.model.train()
        for _ in range(self.epochs):
            order = rng.permutation(len(X))
            for i in range(0, len(X), self.batch):
                idx = order[i:i + self.batch]
                if len(idx) < 2:
                    continue
                xb = (_augment(X[idx], rng) - self.mu) / self.sd
                xb = torch.tensor(xb, dtype=torch.float32, device=self.device)
                yb = torch.tensor(y[idx], dtype=torch.long, device=self.device)
                opt.zero_grad()
                loss_fn(self.model(xb), yb).backward()
                opt.step()
            sched.step()
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        torch = _torch()
        self.model.eval()
        with torch.no_grad():
            xb = torch.tensor((X - self.mu) / self.sd, dtype=torch.float32, device=self.device)
            return torch.softmax(self.model(xb), -1).cpu().numpy()

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.predict_proba(X).argmax(1)

    def save(self, path) -> None:
        torch = _torch()
        torch.save({"state": self.model.state_dict(), "mu": self.mu, "sd": self.sd,
                    "n_classes": self.n_classes}, path)


In [ ]:
%%writefile hbd/train.py
"""Keresztvalidált kiértékelés + végső modellek.

Feladatok (`dataset.TASKS`): binary (lövés / nem lövés), shot3 (felugrásos /
helyből / egyéb), all7 (mind a 7 kategória).

Modellek:
  majority — mindig a leggyakoribb osztály (a viszonyítási alap: ennél
             jobbnak KELL lennie mindennek, különben nem tanult semmit),
  logreg   — logisztikus regresszió a klip-jellemzőkön,
  hgb      — gradiens-erősítéses döntési fák a klip-jellemzőkön,
  tcn      — kis időbeli konvolúciós háló a testpont-sorozaton (torch).

Felosztás: StratifiedGroupKFold a JELENET szerint (ugyanaz a jelenet sosem
lehet egyszerre tanító és teszt). Mérőszámok: pontosság, kiegyensúlyozott
pontosság, makró-F1, osztályonkénti felidézés, tévesztési mátrix.

Kapus-profil: minden lövésre `goalkeeper.shot_profile`; a felugrás-ítélet
a címkével (jump-shot vs shot) összevetve is ki van értékelve.

    python -m hbd.train --poses poses.npz --out results/
"""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

import numpy as np

from .dataset import TASKS
from .features import feature_matrix, sequence
from .goalkeeper import JUMP_MIN_RISE, shot_profile
from .pose import load_dataset

FOLDS = 5


def _sk_models(seed: int):
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    return {
        "majority": lambda: DummyClassifier(strategy="most_frequent"),
        "logreg": lambda: make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5)),
        "hgb": lambda: HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0,
            class_weight="balanced", random_state=seed),
    }


def metrics(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> dict:
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                                 f1_score, recall_score)

    labels = list(range(n_classes))
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "recall_per_class": [float(v) for v in recall_score(
            y_true, y_pred, labels=labels, average=None, zero_division=0)],
        "confusion": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def group_folds(y: np.ndarray, groups: np.ndarray, n_splits: int, seed: int):
    """Rétegzett, csoportos felosztás; kis adatnál kevesebb hajtással."""
    from sklearn.model_selection import StratifiedGroupKFold

    n_groups = len(np.unique(groups))
    min_class = np.bincount(y).min() if len(y) else 0
    k = max(2, min(n_splits, n_groups, max(2, min_class)))
    return list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(
        np.zeros(len(y)), y, groups))


def evaluate_task(task: str, X: np.ndarray, S: np.ndarray, cats: np.ndarray, groups: np.ndarray,
                  seed: int, use_nn: bool, nn_epochs: int, folds: int) -> dict:
    class_names, mapping = TASKS[task]
    y = np.array([mapping[c] for c in cats])
    present = np.unique(y)
    splits = group_folds(y, groups, folds, seed)
    model_makers = dict(_sk_models(seed))
    if use_nn:
        from .models import SeqClassifier

        model_makers["tcn"] = lambda: SeqClassifier(len(class_names), epochs=nn_epochs, seed=seed)

    out = {"classes": list(class_names), "n": int(len(y)),
           "class_counts": np.bincount(y, minlength=len(class_names)).tolist(),
           "folds": len(splits), "models": {}}
    for name, make in model_makers.items():
        pred = np.full(len(y), -1)
        for tr, te in splits:
            m = make()
            if name == "tcn":
                m.fit(S[tr], y[tr])
                pred[te] = m.predict(S[te])
            else:
                m.fit(X[tr], y[tr])
                pred[te] = m.predict(X[te])
        out["models"][name] = metrics(y, pred, len(class_names))
    out["present_classes"] = present.tolist()
    return out


def fit_final(task: str, X, S, cats, out_dir: Path, seed: int, use_nn: bool, nn_epochs: int,
              feature_names: list[str]) -> list[str]:
    """Végső modellek az ÖSSZES adaton (a Macen futtatáshoz)."""
    import joblib

    class_names, mapping = TASKS[task]
    y = np.array([mapping[c] for c in cats])
    saved = []
    hgb = _sk_models(seed)["hgb"]().fit(X, y)
    p = out_dir / f"model_{task}_hgb.joblib"
    joblib.dump({"model": hgb, "classes": list(class_names), "features": feature_names}, p)
    saved.append(p.name)
    if use_nn:
        from .models import SeqClassifier

        nn = SeqClassifier(len(class_names), epochs=nn_epochs, seed=seed).fit(S, y)
        p = out_dir / f"model_{task}_tcn.pt"
        nn.save(p)
        saved.append(p.name)
    return saved


def goalkeeper_report(clips: list[dict], out_dir: Path) -> dict:
    """Kapus-profil minden lövésre + a felugrás-ítélet összevetése a címkével."""
    rows, rises, labels = [], [], []
    for c in clips:
        if c["category"] not in ("jump-shot", "shot"):
            continue
        prof = shot_profile(c["kps"], c["boxes"], c["fps"])
        if prof is None:
            rows.append({"klip": c["name"], "kategoria": c["category"], "itelet": "kevés adat"})
            continue
        rows.append({"klip": c["name"], "kategoria": c["category"], **prof})
        rises.append(prof["csipo_emelkedes"])
        labels.append(int(c["category"] == "jump-shot"))
    if rows:
        keys = sorted({k for r in rows for k in r}, key=lambda k: (k not in ("klip", "kategoria"), k))
        with open(out_dir / "kapus_profilok.csv", "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=keys)
            w.writeheader()
            w.writerows(rows)
    rep: dict = {"loves_db": len(rows), "profil_db": len(rises)}
    if len(set(labels)) == 2:
        from sklearn.metrics import roc_auc_score

        r, lab = np.array(rises), np.array(labels)
        pred = (r >= JUMP_MIN_RISE).astype(int)
        rep["felugras_pontossag"] = float((pred == lab).mean())
        rep["felugras_felidezes_jumpshot"] = float(pred[lab == 1].mean())
        rep["felugras_teves_riasztas_shot"] = float(pred[lab == 0].mean())
        rep["felugras_auc"] = float(roc_auc_score(lab, r))
    prof_rows = [r for r in rows if "kartartas" in r]
    for key in ("kartartas", "lovokez"):
        cnt: dict[str, int] = {}
        for r in prof_rows:
            cnt[r[key]] = cnt.get(r[key], 0) + 1
        rep[f"{key}_eloszlas"] = cnt
    return rep


def _md_table(task_res: dict) -> str:
    lines = ["| modell | pontosság | kiegy. pontosság | makró-F1 | felidézés osztályonként |",
             "|---|---|---|---|---|"]
    for name, m in task_res["models"].items():
        rec = ", ".join(f"{c}: {v:.2f}" for c, v in zip(task_res["classes"], m["recall_per_class"]))
        lines.append(f"| {name} | {m['accuracy']:.3f} | {m['balanced_accuracy']:.3f} | "
                     f"{m['macro_f1']:.3f} | {rec} |")
    best = max((n for n in task_res["models"] if n != "majority"),
               key=lambda n: task_res["models"][n]["macro_f1"], default=None)
    if best:
        cm = task_res["models"][best]["confusion"]
        lines += ["", f"Tévesztési mátrix ({best}; sor = valódi, oszlop = jósolt):", "",
                  "| | " + " | ".join(task_res["classes"]) + " |",
                  "|---" * (len(cm) + 1) + "|"]
        for cname, row in zip(task_res["classes"], cm):
            lines.append(f"| **{cname}** | " + " | ".join(str(v) for v in row) + " |")
    return "\n".join(lines)


def write_report(results: dict, out_dir: Path) -> Path:
    md = ["# UNIRI-HBD — mozdulatfelismerés testpontokból", "",
          f"Klipek: {results['n_clips']}; felosztás: {results['folds']}-szörös, jelenet szerint "
          "csoportosítva (ugyanaz a jelenet sosem tanító és teszt egyszerre).", "",
          "A `majority` a viszonyítási alap: ami nem jobb nála, az nem tanult semmit.", ""]
    for task, res in results["tasks"].items():
        md += [f"## Feladat: {task}", "",
               "Osztályok és darabszámok: " + ", ".join(
                   f"{c} = {n}" for c, n in zip(res["classes"], res["class_counts"])), "",
               _md_table(res), ""]
    gk = results.get("goalkeeper", {})
    if gk:
        md += ["## Kapus-profil (lövésenként, címke nélkül)", "",
               f"Lövések: {gk.get('loves_db')}, ebből profil készült: {gk.get('profil_db')}.", ""]
        if "felugras_pontossag" in gk:
            md += [f"- Felugrás-ítélet a címkéhez mérve (jump-shot vs shot): pontosság "
                   f"{gk['felugras_pontossag']:.3f}, AUC {gk['felugras_auc']:.3f}; felugrásosból "
                   f"felismert {gk['felugras_felidezes_jumpshot']:.2f}, helyből lövésnél tévesen "
                   f"felugrásnak mondott {gk['felugras_teves_riasztas_shot']:.2f}."]
        md += [f"- Kartartás az elengedéskor: {gk.get('kartartas_eloszlas')}",
               f"- Lövőkéz: {gk.get('lovokez_eloszlas')}",
               "- A kartartásra NINCS címke az adathalmazban: kézi mintán kell ellenőrizni "
               "(`kapus_profilok.csv`).", ""]
    if results.get("saved_models"):
        md += ["## Mentett végső modellek", ""] + [f"- `{p}`" for p in results["saved_models"]]
    p = out_dir / "results.md"
    p.write_text("\n".join(md) + "\n")
    return p


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description="UNIRI-HBD: kiértékelés és tanítás")
    ap.add_argument("--poses", required=True, help="a hbd.pose kimenete (.npz)")
    ap.add_argument("--out", required=True, help="kimeneti mappa")
    ap.add_argument("--tasks", default="binary,shot3,all7")
    ap.add_argument("--folds", type=int, default=FOLDS)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no-nn", action="store_true", help="a torch-háló kihagyása")
    ap.add_argument("--nn-epochs", type=int, default=80)
    ap.add_argument("--no-final", action="store_true", help="ne tanítson végső modellt")
    args = ap.parse_args(argv)

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    clips = [c for c in load_dataset(args.poses)["clips"] if len(c["kps"]) > 0]
    X, feat_names = feature_matrix(clips)
    S = np.stack([sequence(c["kps"], c["boxes"]) for c in clips])
    cats = np.array([c["category"] for c in clips])
    groups = np.array([c["scene_id"] for c in clips])
    use_nn = not args.no_nn
    if use_nn:
        try:
            import torch  # noqa: F401
        except ImportError:
            use_nn = False

    results = {"n_clips": len(clips), "folds": args.folds, "features": feat_names, "tasks": {}}
    saved = []
    for task in [t for t in args.tasks.split(",") if t]:
        print(f"== {task} ==", flush=True)
        res = evaluate_task(task, X, S, cats, groups, args.seed, use_nn, args.nn_epochs, args.folds)
        results["tasks"][task] = res
        for name, m in res["models"].items():
            print(f"  {name:9s} pontosság {m['accuracy']:.3f}  kiegy. {m['balanced_accuracy']:.3f}"
                  f"  makró-F1 {m['macro_f1']:.3f}", flush=True)
        if not args.no_final:
            saved += fit_final(task, X, S, cats, out, args.seed, use_nn, args.nn_epochs, feat_names)
    results["goalkeeper"] = goalkeeper_report(clips, out)
    results["saved_models"] = saved
    (out / "results.json").write_text(json.dumps(results, ensure_ascii=False, indent=1))
    p = write_report(results, out)
    print(f"Jelentés: {p}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/release.py
"""Tanított lövés-profil: elengedés pillanata, kartartás, lövőkéz — kézi jelölésből.

A `goalkeeper` kézi küszöbei (a "leggyorsabb csukló" kockája) rosszul
találták meg az elengedést, ezért a kartartást is elrontották (a lövések
felét "lent"-nek mondták, a jelölés szerint 337-ből 2 az). Itt ugyanezt a
gép tanulja meg a lövésjelölő oldalon készült kézi jelölésekből:

  1. ELENGEDÉS: kis időbeli konvolúciós háló kockánként pontoz, a legjobb
     kocka az elengedés (tanítás: a jelölt kocka körüli puha célfüggvény).
  2. KARTARTÁS (fent / oldalt) és LÖVŐKÉZ (jobb / bal): logisztikus
     regresszió a testtartás-jellemzőkön az elengedés körüli ablakban.

Kiértékelés: jelenet szerint csoportosított keresztvalidáció, "végponttól
végpontig" — a teszt-részen a kartartást a MODELL által talált elengedésnél
mérjük, nem a kézi jelölésnél (úgy, ahogy élesben futna). Viszonyítás: a
régi szabály és a "mindig a leggyakoribb" tipp.

    python -m hbd.release --poses poses.npz --labels cimkek.csv --out results_release/
"""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

import numpy as np

from .features import (L_EL, L_SH, L_WR, NOSE, R_EL, R_SH, R_WR, normalize)
from .goalkeeper import shot_profile
from .pose import load_dataset

# Az elengedés körüli ablak a kartartás-jellemzőkhöz — IDŐTARTAM, ezért
# másodpercben (a jelenetek 30 és 48 fps-esek).
ARM_WINDOW_S = 0.08
# A puha célfüggvény szélessége (másodperc): a jelölés ±1-2 kockás
# bizonytalanságát nem büntetjük teljes hibaként.
TARGET_SIGMA_S = 0.03
# Kartartás-osztályok: a "lent" a jelölésben 337-ből 2 — ennyiből nem
# tanulható, az "oldalt"-hoz soroljuk (mindkettő: a kar a fej vonala alatt).
ARM_CLASSES = ("fent", "oldalt")
HAND_CLASSES = ("jobb", "bal")
FOLDS = 5


# ---------------------------------------------------------------- jelölések
def load_labels(path: str | Path) -> dict[str, dict]:
    """A jelölések CSV-ből (klip → sor); a "rossz" klipek kimaradnak."""
    out = {}
    with open(path) as fh:
        for r in csv.DictReader(fh):
            if r["rossz"].lower() in ("true", "1"):
                continue
            out[r["klip"]] = {"elengedes": int(r["elengedes"]), "kartartas": r["kartartas"],
                              "lovokez": r["lovokez"], "kategoria": r["kategoria"]}
    return out


# --------------------------------------------------------------- jellemzők
def frame_features(kps: np.ndarray, boxes: np.ndarray, fps: float) -> np.ndarray:
    """Kockánkénti bemenet az elengedés-hálónak: (T, F) float32.

    Testpontok a csípőhöz és törzshosszhoz normálva, a csuklók sebessége
    (egység / s) és magassága a vállhoz, plusz a test elmozdulása.
    """
    fps = fps or 30.0
    rel, motion, conf, _ = normalize(kps, boxes)
    T = len(kps)
    vel = np.zeros((T, 17, 2))
    if T > 1:
        vel[1:] = np.diff(rel, axis=0) * fps
    wr_speed = np.linalg.norm(vel[:, [L_WR, R_WR]], axis=-1) / 10.0
    wr_h = np.stack([rel[:, L_WR, 1] - rel[:, L_SH, 1], rel[:, R_WR, 1] - rel[:, R_SH, 1]], 1)
    mvel = np.zeros((T, 2))
    if T > 1:
        mvel[1:] = np.diff(motion, axis=0) * fps / 10.0
    return np.concatenate([rel.reshape(T, -1), vel.reshape(T, -1) / 10.0, wr_speed, wr_h,
                           motion, mvel, conf], 1).astype(np.float32)


def arm_hand_features(kps: np.ndarray, boxes: np.ndarray, fps: float, r: int) -> np.ndarray:
    """Testtartás az elengedés (r) körüli ablakban → jellemző-vektor."""
    fps = fps or 30.0
    rel, _, _, _ = normalize(kps, boxes)
    T = len(kps)
    w = max(1, int(round(ARM_WINDOW_S * fps)))
    a, b = max(0, r - w), min(T, r + w + 1)
    seg = rel[a:b]
    # a törzs "fel" iránya (csípő-közép → váll-közép); a kar szöge ehhez mérve:
    # fent ≈ 0–40°, oldalt ≈ 60–110° (a "sidearm" dobás lényege a szög, nem a magasság)
    up = (seg[:, L_SH] + seg[:, R_SH]) / 2
    up = up / (np.linalg.norm(up, axis=1, keepdims=True) + 1e-9)

    def ang(v):
        v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
        return np.degrees(np.arccos(np.clip((v * up).sum(1), -1, 1)))

    f = []
    for wr, sh, el in ((L_WR, L_SH, L_EL), (R_WR, R_SH, R_EL)):
        h = seg[:, wr, 1] - seg[:, sh, 1]
        head = seg[:, wr, 1] - seg[:, NOSE, 1]
        side = np.abs(seg[:, wr, 0] - seg[:, sh, 0])
        elb = seg[:, el, 1] - seg[:, sh, 1]
        a_arm = ang(seg[:, wr] - seg[:, sh])
        a_upper = ang(seg[:, el] - seg[:, sh])
        f += [h.max(), h.mean(), head.max(), side.mean(), elb.max(),
              a_arm.min(), a_arm.mean(), a_upper.min(), a_upper.mean()]
    # a teljes klipen a csuklók legnagyobb magassága és sebessége (a lövőkéz
    # a lendítés során magasabbra és gyorsabban mozog)
    hl = rel[:, L_WR, 1] - rel[:, L_SH, 1]
    hr = rel[:, R_WR, 1] - rel[:, R_SH, 1]
    vl = np.linalg.norm(np.diff(rel[:, L_WR], axis=0), axis=1) * fps if T > 1 else np.zeros(1)
    vr = np.linalg.norm(np.diff(rel[:, R_WR], axis=0), axis=1) * fps if T > 1 else np.zeros(1)
    f += [hl.max() - hr.max(), np.percentile(vl, 90) - np.percentile(vr, 90),
          np.abs(seg[:, L_SH, 0] - seg[:, R_SH, 0]).mean()]
    return np.nan_to_num(np.array(f, np.float64))


# ------------------------------------------------------------ elengedés-háló
def _torch():
    import torch

    return torch


class ReleaseNet:
    """Kockánként pontozó időbeli konvolúciós háló; predict → elengedés kockája."""

    def __init__(self, epochs: int = 120, lr: float = 2e-3, seed: int = 0, width: int = 64):
        self.epochs, self.lr, self.seed, self.width = epochs, lr, seed, width
        self.model = None
        self.mu = self.sd = None
        torch = _torch()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def _build(self, f_in: int):
        torch = _torch()
        nn = torch.nn
        w = self.width
        return nn.Sequential(
            nn.Conv1d(f_in, w, 5, padding=2), nn.ReLU(),
            nn.Conv1d(w, w, 5, padding=4, dilation=2), nn.ReLU(),
            nn.Conv1d(w, w, 5, padding=8, dilation=4), nn.ReLU(),
            nn.Dropout(0.2), nn.Conv1d(w, 1, 1),
        ).to(self.device)

    def _batch(self, X: list[np.ndarray]):
        torch = _torch()
        T = max(len(x) for x in X)
        xb = np.zeros((len(X), T, X[0].shape[1]), np.float32)
        mask = np.zeros((len(X), T), bool)
        for i, x in enumerate(X):
            xb[i, :len(x)] = (x - self.mu) / self.sd
            mask[i, :len(x)] = True
        return (torch.tensor(xb, device=self.device).transpose(1, 2),
                torch.tensor(mask, device=self.device))

    def fit(self, X: list[np.ndarray], y: list[int], fps: list[float]):
        torch = _torch()
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        allx = np.concatenate(X)
        self.mu, self.sd = allx.mean(0), allx.std(0) + 1e-6
        self.model = self._build(allx.shape[1])
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-3)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, self.epochs)
        self.model.train()
        for _ in range(self.epochs):
            for idx in np.array_split(rng.permutation(len(X)), max(1, len(X) // 16)):
                xb, mask = self._batch([X[i] for i in idx])
                logits = self.model(xb).squeeze(1).masked_fill(~mask, -1e9)
                tgt = np.zeros(mask.shape, np.float32)
                t = np.arange(mask.shape[1])
                for j, i in enumerate(idx):
                    sig = max(0.5, TARGET_SIGMA_S * fps[i])
                    g = np.exp(-0.5 * ((t - y[i]) / sig) ** 2) * mask[j].cpu().numpy()
                    tgt[j] = g / g.sum()
                tgt_t = torch.tensor(tgt, device=self.device)
                loss = -(tgt_t * torch.log_softmax(logits, 1)).sum(1).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()
        return self

    def predict(self, X: list[np.ndarray]) -> np.ndarray:
        torch = _torch()
        self.model.eval()
        out = []
        with torch.no_grad():
            for x in X:
                xb, mask = self._batch([x])
                out.append(int(self.model(xb).squeeze(1)[0, :len(x)].argmax()))
        return np.array(out)

    def save(self, path) -> None:
        torch = _torch()
        torch.save({"state": self.model.state_dict(), "mu": self.mu, "sd": self.sd,
                    "width": self.width}, path)


# ---------------------------------------------------------------- kiértékelés
def _clf():
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5))


def evaluate(clips: list[dict], labels: dict[str, dict], seed: int = 0, epochs: int = 120) -> dict:
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.model_selection import StratifiedGroupKFold

    data = [c for c in clips if c["name"] in labels and len(c["kps"]) >= 8]
    y_rel = np.array([min(labels[c["name"]]["elengedes"], len(c["kps"]) - 1) for c in data])
    y_arm = np.array([int(labels[c["name"]]["kartartas"] != "fent") for c in data])
    y_hand = np.array([int(labels[c["name"]]["lovokez"] == "bal") for c in data])
    fps = np.array([c["fps"] for c in data])
    groups = np.array([c["scene_id"] for c in data])
    F = [frame_features(c["kps"], c["boxes"], c["fps"]) for c in data]

    pred_rel = np.zeros(len(data), int)
    pred_arm = np.zeros(len(data), int)
    pred_hand = np.zeros(len(data), int)
    oracle_arm = np.zeros(len(data), int)
    strat = y_arm * 2 + y_hand
    for tr, te in StratifiedGroupKFold(FOLDS, shuffle=True, random_state=seed).split(
            np.zeros(len(data)), strat, groups):
        net = ReleaseNet(epochs=epochs, seed=seed).fit([F[i] for i in tr], y_rel[tr], fps[tr])
        pred_rel[te] = net.predict([F[i] for i in te])
        Xtr = np.stack([arm_hand_features(data[i]["kps"], data[i]["boxes"], fps[i], y_rel[i]) for i in tr])
        Xte_pred = np.stack([arm_hand_features(data[i]["kps"], data[i]["boxes"], fps[i], pred_rel[i]) for i in te])
        Xte_orc = np.stack([arm_hand_features(data[i]["kps"], data[i]["boxes"], fps[i], y_rel[i]) for i in te])
        arm = _clf().fit(Xtr, y_arm[tr])
        hand = _clf().fit(Xtr, y_hand[tr])
        pred_arm[te] = arm.predict(Xte_pred)
        oracle_arm[te] = arm.predict(Xte_orc)
        pred_hand[te] = hand.predict(Xte_pred)

    # a régi kézi szabály ugyanezeken a klipeken
    rule_rel, rule_arm, rule_hand = [], [], []
    for c in data:
        p = shot_profile(c["kps"], c["boxes"], c["fps"])
        rule_rel.append(p["elengedes_kocka"] if p else len(c["kps"]) // 2)
        rule_arm.append(int(not p or p["kartartas"] != "fent"))
        rule_hand.append(int(bool(p) and p["lovokez"] == "bal"))
    rule_rel, rule_arm, rule_hand = map(np.array, (rule_rel, rule_arm, rule_hand))

    def rel_stats(pred):
        err_f = np.abs(pred - y_rel)
        err_ms = err_f / fps * 1000
        return {"atlag_hiba_kocka": float(err_f.mean()), "median_hiba_ms": float(np.median(err_ms)),
                "pontos_2_kockan_belul": float((err_f <= 2).mean()),
                "pontos_100ms_belul": float((err_ms <= 100).mean())}

    def cls_stats(pred, y):
        return {"pontossag": float((pred == y).mean()),
                "kiegyensulyozott": float(balanced_accuracy_score(y, pred))}

    return {
        "klipek": len(data),
        "osztalyok": {"kartartas": {"fent": int((y_arm == 0).sum()), "oldalt/lent": int(y_arm.sum())},
                      "lovokez": {"jobb": int((y_hand == 0).sum()), "bal": int(y_hand.sum())}},
        "elengedes": {"modell": rel_stats(pred_rel), "regi_szabaly": rel_stats(rule_rel)},
        "kartartas": {"modell": cls_stats(pred_arm, y_arm),
                      "modell_kezi_elengedesnel": cls_stats(oracle_arm, y_arm),
                      "regi_szabaly": cls_stats(rule_arm, y_arm),
                      "mindig_fent": cls_stats(np.zeros_like(y_arm), y_arm)},
        "lovokez": {"modell": cls_stats(pred_hand, y_hand),
                    "regi_szabaly": cls_stats(rule_hand, y_hand),
                    "mindig_jobb": cls_stats(np.zeros_like(y_hand), y_hand)},
    }


def fit_final(clips, labels, out: Path, seed: int = 0, epochs: int = 120) -> list[str]:
    import joblib

    data = [c for c in clips if c["name"] in labels and len(c["kps"]) >= 8]
    y_rel = np.array([min(labels[c["name"]]["elengedes"], len(c["kps"]) - 1) for c in data])
    fps = np.array([c["fps"] for c in data])
    F = [frame_features(c["kps"], c["boxes"], c["fps"]) for c in data]
    ReleaseNet(epochs=epochs, seed=seed).fit(F, y_rel, fps).save(out / "release_net.pt")
    X = np.stack([arm_hand_features(c["kps"], c["boxes"], c["fps"], r) for c, r in zip(data, y_rel)])
    y_arm = np.array([int(labels[c["name"]]["kartartas"] != "fent") for c in data])
    y_hand = np.array([int(labels[c["name"]]["lovokez"] == "bal") for c in data])
    joblib.dump({"model": _clf().fit(X, y_arm), "classes": ARM_CLASSES}, out / "arm_clf.joblib")
    joblib.dump({"model": _clf().fit(X, y_hand), "classes": HAND_CLASSES}, out / "hand_clf.joblib")
    return ["release_net.pt", "arm_clf.joblib", "hand_clf.joblib"]


def write_report(res: dict, out: Path) -> Path:
    e, k, h = res["elengedes"], res["kartartas"], res["lovokez"]
    pct = lambda v: f"{100 * v:.1f}%"  # noqa: E731
    md = [
        "# Tanított lövés-profil (kézi jelölésből)", "",
        f"Klipek: {res['klipek']} (a „rossz” klipek nélkül); 5-szörös, jelenet szerint "
        "csoportosított keresztvalidáció. A kartartást és a lövőkezet a MODELL által talált "
        "elengedésnél mérjük, ahogy élesben futna.", "",
        f"Osztályok: kartartás {res['osztalyok']['kartartas']}, lövőkéz {res['osztalyok']['lovokez']}.", "",
        "## Elengedés pillanata", "",
        "| | átlagos hiba (kocka) | medián hiba | ±2 kockán belül | 100 ms-on belül |",
        "|---|---|---|---|---|",
    ]
    for name, key in (("tanított modell", "modell"), ("régi szabály", "regi_szabaly")):
        s = e[key]
        md.append(f"| {name} | {s['atlag_hiba_kocka']:.1f} | {s['median_hiba_ms']:.0f} ms | "
                  f"{pct(s['pontos_2_kockan_belul'])} | {pct(s['pontos_100ms_belul'])} |")
    md += ["", "## Kartartás (fent / oldalt)", "", "| | pontosság | kiegyensúlyozott |", "|---|---|---|"]
    for name, key in (("tanított modell", "modell"), ("modell a kézi elengedésnél", "modell_kezi_elengedesnel"),
                      ("régi szabály", "regi_szabaly"), ("mindig „fent”", "mindig_fent")):
        md.append(f"| {name} | {pct(k[key]['pontossag'])} | {pct(k[key]['kiegyensulyozott'])} |")
    md += ["", "## Lövőkéz (jobb / bal)", "", "| | pontosság | kiegyensúlyozott |", "|---|---|---|"]
    for name, key in (("tanított modell", "modell"), ("régi szabály", "regi_szabaly"),
                      ("mindig „jobb”", "mindig_jobb")):
        md.append(f"| {name} | {pct(h[key]['pontossag'])} | {pct(h[key]['kiegyensulyozott'])} |")
    p = out / "release_results.md"
    p.write_text("\n".join(md) + "\n")
    return p


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Tanított lövés-profil a kézi jelölésből")
    ap.add_argument("--poses", required=True)
    ap.add_argument("--labels", required=True, help="a lövésjelölő CSV-je")
    ap.add_argument("--out", required=True)
    ap.add_argument("--epochs", type=int, default=120)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args(argv)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    clips = load_dataset(args.poses)["clips"]
    labels = load_labels(args.labels)
    res = evaluate(clips, labels, args.seed, args.epochs)
    res["mentett_modellek"] = fit_final(clips, labels, out, args.seed, args.epochs)
    (out / "release_results.json").write_text(json.dumps(res, ensure_ascii=False, indent=1))
    print(write_report(res, out).read_text())
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 1. Testpont-kinyerés (GPU)

In [ ]:
from hbd.dataset import find_root, index_dataset
DATA = str(find_root('/kaggle/input'))
print('adat:', DATA, '| klipek:', len(index_dataset(DATA)))

In [ ]:
!python -m hbd.pose --data "$DATA" --out /kaggle/working/poses.npz --device 0 --preview 6

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/poses_preview.jpg'))

## 2. Kiértékelés és végső modellek

In [ ]:
!python -m hbd.train --poses /kaggle/working/poses.npz --out /kaggle/working/results

In [ ]:
from IPython.display import Markdown
display(Markdown(open('/kaggle/working/results/results.md').read()))